# 调用线上的大模型

## 1. 调用具体模型厂商的API

### 1.1 调用DeepSeek官网的大模型

调用大模型涉及到三个特别重要的参数：base_url、model_name、api_key

举例1：将参数信息存储在.env配置文件中

In [2]:
import os

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain_openai import ChatOpenAI

# 1. 读取模型的配置文件
# 通过load_dotenv()将.env中的变量加载为环境变量
# override=True表示：无论你当前的操作系统、终端或者虚拟环境中是否已经存在同名的环境变量，都会强行用 .env 文件里写的值去覆盖它
load_dotenv(override=True)
#从本地配置文件里获取配置
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = os.getenv('DEEPSEEK_BASE_URL')

# 2.大模型的初始化，返回的是一个
llm_deepseek = ChatDeepSeek(
    model='deepseek-flash',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

# 3. 大模型的调用
response = llm_deepseek.invoke('请用一句话介绍你自己')

print(response)

content='我是一个乐于助人的AI助手，可以回答问题、处理信息并协助你完成各种任务。' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文请求：“请用一句话介绍你自己”。需要一句话介绍自己。作为AI助手。必须一句话。可以简洁。注意需要符合。不要多余。可以：“我是由深度求索公司开发的AI助手，名叫DeepSeek，能帮你解答问题、处理信息和提供创意。” 这是一句话。可能用户期望用一句话介绍。我们就用一句话。要确保不声称未给定身份？系统无指定名字，但通常我是DeepSeek AI？在当前环境，我们是API助手，可能被要求介绍自己。可以说“我是一个乐于助人的AI助手，可以回答问题、处理信息并协助你完成各种任务。” 这一句即可，避免公司名。用户用中文。回答一句话。确保只有一个句号。 final. '} response_metadata={'token_usage': {'completion_tokens': 175, 'prompt_tokens': 35, 'total_tokens': 210, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 155, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'd6ef6cb7-4826-426b-9a4d-7c9433fb0669', 'finish_reason': 'stop'

方式2：优化，依靠默认行为读取 .env 环境变量

In [3]:
# 1. 加载配置文件
load_dotenv(override=True)

# 2. 由于配置文件的默认名称和langchain_deepseek下的chat_models.py的配置一致，所以可以直接获取，不用重新获取
llm_deepseek = ChatDeepSeek(
    model='deepseek-flash',
)

# 3. 打印调用大模型后的返回结果
print(llm_deepseek.invoke('请用一句话介绍你自己').content)


我是由深度求索公司创造的AI助手，乐于为你提供各种问题的解答与帮助。


## 1.2 调用通义千问的大模型
通过阿里云百炼平台调用，官网：https://bailian.console.aliyun.com/

In [34]:
from dotenv import load_dotenv
import dashscope
import os
#1. 优先加载配置
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv('DASHSCOPE_API_KEY')
DASHSCOPE_BASE_URL = os.getenv('DASHSCOPE_BASE_URL')
dashscope.base_compatible_api_url = DASHSCOPE_BASE_URL
#2. 通义模型的配置，注意配置文件和python的class配置保持一致，就不需要自己手动加载，直接读取就行,注意可行的模型是：qwen-flash,其他的模型不行
# 如果要可以的话，就需要用其他的方式

msg = [
    {'role': 'user',
     'content': '请用一句话介绍你自己'}
]
response = dashscope.MultiModalConversation.call(
    model='qwen3.7-flash',
    api_key=DASHSCOPE_API_KEY,
    messages=msg,
)

#直接读取返回结果
print(response.output.choices[0].message.content[0]["text"])


我是Qwen（通义千问），由阿里巴巴集团旗下通义实验室自主研发的大语言模型，致力于为您提供准确、可靠且有价值的智能协助。


## 2. 兼容写法（采用OpenAI的接口调用）
一方面，LangChain没有为所有大模型厂商提供专用接口。如果选用的平台没有专用接口，可以通过兼容接口调用。另一方面，专用接口的对接方式五花八门，如腾讯混元的ChatHunyuan需要单独的 APP_ID + SecretId SecretKey ，配置繁琐，用户不友好。

**结论：大多数API平台都支持OpenAI API接口规范，所以基本都可以通过 ChatOpenAI 集成。**

举例：将DeepSeek的大模型改成OpenAI调用

In [37]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

load_dotenv(override=True)

#1. 采用DeepSeek的OpenAI写法
#DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
#DEEPSEEK_BASE_URL = os.getenv('DEEPSEEK_BASE_URL')

# 采用ChatOpenAI
# llm_deepseek = ChatOpenAI(
#     model='deepseek-flash',
#     api_key=DEEPSEEK_API_KEY,
#     base_url=DEEPSEEK_BASE_URL,
# )

#2. 采用通义千问的OpenAI写法
DASHSCOPE_API_KEY = os.getenv('DASHSCOPE_API_KEY')
DASHSCOPE_BASE_URL = os.getenv('DASHSCOPE_BASE_URL')

# 采用ChatOpenAI
llm_openai = ChatOpenAI(
    model='qwen3.7-flash',
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
)

print(llm_openai.invoke('一句话介绍下你自己').content)

我是通义千问（Qwen），一个真诚友好、乐于助人的 AI 伙伴，期待能用清晰的思路和温暖的交流为你答疑解惑、激发灵感。


## 3. 使用 LangChain 1.x的统一方式
init_chat_model是LangChain 1.x中推出的，用于初始化聊天模型的**统一接口**。只要LangChain支持的模型都可以处理，它会根据模型名称自动选择对应的模型类初始化实例。

基本语法：

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model='qwen3.7-flash',
    model_provider='Qwen',   #模型提供商名称
    api_key='<your-api-key>',
    configurable_fields=["temperature"],  # 只允许动态修改 temperature
    temperature=0.7  # 范围通常是 0 ~ 2
)

问题：init_chat_model和直接使用ChatTongyi、ChatOpenAI、ChatDeepSeek等有什么区别？

回答：init_chat_model是LangChain 1.x的统一接口，优势包括：

（1）统一接口：无需记住每个提供商的不同初始化方式（以一致的方式初始化）

（2）易于切换：简化了智能体系统中模型切换策略（只需修改模型字符串）

（3）简洁明了：更简洁的语法，减少样板代码

（4）自动适配：内部根据模型标识自动选择对应的驱动类（ChatTongyi、ChatOpenAI、ChatDeepSeek）


In [47]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1. 采用阿里通义千问的大模型
# API_KEY = os.getenv('DASHSCOPE_API_KEY')
# BASE_URL = os.getenv('DASHSCOPE_BASE_URL')

#注意，调用千问模型的时候，用openai的厂家。地址也需要采用OpenAI的。
# llm_model = init_chat_model(
#     model='qwen3.7-flash',
#     model_provider='openai',  #关键点之一
#     api_key=API_KEY,
#     base_url=BASE_URL, # 关键点之二
#     temperature=0.7
# )

# 2. 采用DeepSeek的大模型调用
API_KEY = os.getenv('DEEPSEEK_API_KEY')
BASE_URL = os.getenv('DEEPSEEK_BASE_URL')

llm_model = init_chat_model(
    model='deepseek-flash',
    model_provider='deepseek',  #关键点之一
    api_key=API_KEY,
    base_url=BASE_URL, # 关键点之二
    temperature=0.7
)

print(llm_model.invoke('一句话介绍下你自己').content)

我是由深度求索公司创造的AI助手，可以帮你解答问题、提供信息并处理各种任务。


#### 问题1：model_provider支持哪些provider？

model_provider 表示模型的提供者，支持的providers有： anthropic , anthropic_bedrock,azure_ai, azure_openai, bedrockbedrock_converse, cohere, deepseek , fireworks,google_anthropic_vertex, google_genai, google_vertexaigrog, huggingface, ibm, mistralai, nvidia,ollama , openai , openrouter , perplexity, together, upstage, xai。

如果 model_provider="openai" ，会自动加载 langchain-openai 的依赖包，底层调用的是ChatOpenAI 类。

如果 model_provider="deepseek" ，会自动加载 langchain-deepseek 的依赖包，底层调用的是 ChatDeepSeek 类。

像阿里的 dashscope 尚未被LangChain官方纳入模型的统一注册体系，暂时不知道"dashscope"的提供者是谁。此时可以将model_provider设置为openai，底层将会用openai的规范处理请求，这就要求我们调用的模型服务是OpenAI Compatible的。

#### 问题2：如果在model参数中没有指明模型提供者，必须在model_provider中指明？
可以在model参数中通过前缀指定模型供应商，和模型名称之间用 冒号分割 ，等价于通过model_provider参数指定供应商。如果两个位置都没有指明供应商，LangChain底层会按照内置规则自动推断。

但是，并非所有的模型都支持自动推断，如model名称 qwen-plus 不支持自动推断，没有指明供应商会报错。